In [ ]:
import pandas as pd
from fxml.db.connection import connect
from fxml.db.repository import load_bars

conn = connect()
h1 = load_bars(conn, "EURUSD", "H1")
print(h1.shape)
print(h1.index.min()), h1.index.max()
h1.head()

In [ ]:
high_ok = (h1["high"] >= h1[["open", "close"]].max(axis=1)).all()
low_ok = (h1["low"] <= h1[["open", "close"]].min(axis=1)).all()
print("high >= open/close everywhere:", high_ok)
print("low  <= open/close everywhere:", low_ok)
print("duplicate timestamps:", h1.index.duplicated().sum())
print("zero or negative prices:", (h1[["open", "high", "low", "close"]] <= 0).any().any())
h1.describe()


In [ ]:
step = h1.index.to_series().diff().dropna()
gaps = step[step > pd.Timedelta(hours=1)]
print("total gaps:", len(gaps))
print(gaps.value_counts().head(12))
print("\nUnexpected gaps (2h to 40h, so not a normal weekend):")
print(gaps[(gaps > pd.Timedelta(hours=2)) & (gaps < pd.Timedelta(hours=40))])


In [ ]:
h1["close"].plot(figsize=(12, 3), title="EURUSD H1 close")


In [ ]:
spread_by_year = h1["spread"].groupby(h1.index.year).median()
print(spread_by_year.to_string())
